# Wheel-Based Pre-Oracle Filtering Tradeoff

This notebook compares `mod30`, `mod210`, and `mod2310` as optional classical pre-oracle filters before Quantum Oracle Sketching (QOS).

Core idea:

```text
raw classical candidates
    → wheel filter
    → filtered candidates
    → QOS sampling / oracle construction
```

The notebook measures candidate fraction and a simple oracle-call proxy. It does **not** claim a quantum speedup; it isolates the classical filtering layer.

In [ ]:
import sys
from pathlib import Path

# Run from repo root. If opened in Colab, clone/upload repo first.
sys.path.append(str(Path.cwd()))

from modwheel import STANDARD_WHEELS
from pre_oracle_filter import oracle_call_proxy, pre_oracle_candidates_by_name

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Wheel density table

A wheel modulus is the product of small primes. Admissible residues are residues coprime to the modulus.

In [ ]:
rows = []
for name, wheel in STANDARD_WHEELS.items():
    rows.append({
        "wheel": name,
        "primes": "·".join(map(str, wheel.primes)),
        "modulus": wheel.modulus,
        "residue_count": wheel.residue_count,
        "density": wheel.density,
        "reduction": wheel.reduction,
    })

df = pd.DataFrame(rows)
df

## 2. Oracle-call proxy experiment

We treat each candidate entering oracle construction as one proxy oracle input. This is a simplified workload proxy, useful for comparing pre-oracle filter depth.

In [ ]:
N = 100_000
values = range(1, N + 1)
baseline = oracle_call_proxy(values)

bench_rows = []
for name, wheel in STANDARD_WHEELS.items():
    remaining = oracle_call_proxy(range(1, N + 1), wheel)
    bench_rows.append({
        "wheel": name,
        "baseline_candidates": baseline,
        "remaining": remaining,
        "remaining_fraction": remaining / baseline,
        "reduction_fraction": 1 - remaining / baseline,
    })

bench = pd.DataFrame(bench_rows)
bench

## 3. Candidate fraction figure

This is the main paper/README figure: deeper wheels reduce the candidate stream, but with diminishing returns after `mod30`.

In [ ]:
fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

names = bench["wheel"].tolist()
fractions = bench["remaining_fraction"].to_numpy()

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(names, fractions, marker="o", linewidth=2)
ax.set_title("Wheel-Based Pre-Oracle Filtering: Diminishing Returns")
ax.set_xlabel("Wheel filter")
ax.set_ylabel("Remaining candidate fraction")
ax.set_ylim(0, 0.32)
ax.grid(True, alpha=0.35)

labels = [
    "8/30 = 26.7%\n73.3% excluded",
    "48/210 = 22.9%\n77.1% excluded",
    "480/2310 = 20.8%\n79.2% excluded",
]
for x, y, label in zip(names, fractions, labels):
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 10), ha="center")

fig.tight_layout()
svg_path = fig_dir / "modwheel_pre_oracle_tradeoff.svg"
png_path = fig_dir / "modwheel_pre_oracle_tradeoff.png"
fig.savefig(svg_path)
fig.savefig(png_path, dpi=200)
plt.show()

svg_path, png_path

## 4. Optional row-id adapter sketch for `real_datasets/`

For real datasets, the safest first integration is to filter row/sample IDs before passing data to QOS scripts. This preserves QOS internals and keeps the wheel layer optional.

In [ ]:
# Example adapter sketch only.
# Assume X and y are already loaded by a real_datasets script.

def filter_dataset_by_row_ids(X, y, wheel_name="mod30"):
    """Filter rows by wheel-admissible sample indices."""
    row_ids = range(len(X))
    filtered_ids = pre_oracle_candidates_by_name(row_ids, wheel_name)
    return X[filtered_ids], y[filtered_ids], filtered_ids

# Usage inside a dataset script would look like:
# X_filtered, y_filtered, ids = filter_dataset_by_row_ids(X, y, "mod30")

## 5. Interpretation

- `mod30` keeps about `26.7%` of candidates.
- `mod210` keeps about `22.9%`.
- `mod2310` keeps about `20.8%`.

Most candidate reduction occurs at `mod30`; deeper wheels add smaller reductions. The useful research question is therefore a tradeoff:

```text
classical filtering cost vs oracle construction / query cost
```

This notebook provides the first reproducible figure for that tradeoff.